# Sprint 3 — Codificação de Mecanismos de Atenção

Projeto Integrador — Construção de um Large Language Model (LLM) From Scratch

Baseado no Capítulo 3 (*Coding Attention Mechanisms*) do livro **Build a Large Language Model (From Scratch)**, de Sebastian Raschka.

Este notebook reúne, em sequência, todos os códigos feitos nos arquivos `src/Sprint 3/3,1.py` a `3,4.py`, seguindo a mesma lógica da Sprint 2: primeiro o exemplo do livro (para validar contra os valores impressos no próprio livro), depois a aplicação do mesmo mecanismo do projeto (`the-verdict.txt`, via BPE do GPT-2), reaproveitando os mesmos Token IDs usados desde a Sprint 2 (`[290, 4920, 2241, 287]`).

## 3.3 — Autoatenção simplificada (sem pesos treináveis)

Primeira versão do mecanismo de atenção, sem nenhum peso treinável. Serve só para fixar a mecânica: escore de atenção (produto escalar), normalização (softmax) e vetor de contexto (soma ponderada). Frase de exemplo do livro, já embeddada em vetores de 3 dimensões.

In [1]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print(attn_weights_2)
print(attn_weights_2.sum())

context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print(context_vec_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)
tensor([0.4419, 0.6515, 0.5683])


In [2]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
all_context_vecs = torch.matmul(attn_weights, inputs)

print(attn_weights.sum(dim=-1))
print(all_context_vecs)

print(context_vec_2)
print(all_context_vecs[1])

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])


### Aplicando 

In [4]:
from pathlib import Path
import tiktoken

BASE_DIR = Path.cwd().resolve().parents[0]
with open(BASE_DIR / "data" / "the-verdict.txt", "r", encoding="utf-8") as f:
    texto_completo = f.read()

tokenizer = tiktoken.get_encoding("gpt2")
enc_text = tokenizer.encode(texto_completo)
print(len(enc_text))

entrada_real = torch.tensor([290, 4920, 2241, 287])
print([tokenizer.decode([tid]) for tid in entrada_real.tolist()])

vocab_size = 50257
output_dim = 256
context_length_real = 4

torch.manual_seed(123)
camada_embedding = torch.nn.Embedding(vocab_size, output_dim)
camada_posicional = torch.nn.Embedding(context_length_real, output_dim)

token_embeddings = camada_embedding(entrada_real)
pos_embeddings = camada_posicional(torch.arange(context_length_real))
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

5146
[' and', ' established', ' himself', ' in']
torch.Size([4, 256])


In [5]:
attn_scores_real = input_embeddings @ input_embeddings.T
attn_weights_real = torch.softmax(attn_scores_real, dim=-1)
print(attn_weights_real.sum(dim=-1))

context_vecs_real = attn_weights_real @ input_embeddings
print(context_vecs_real.shape)
print(context_vecs_real)

tensor([1., 1., 1., 1.], grad_fn=<SumBackward1>)
torch.Size([4, 256])
tensor([[-0.7813,  1.0511, -2.9067,  ...,  1.3991, -1.7625,  1.5602],
        [ 0.6601, -0.1455, -2.7764,  ..., -1.5066,  1.1014, -1.4168],
        [ 1.5837, -0.2107,  2.2676,  ...,  1.6770,  0.5643, -2.0603],
        [ 1.4003,  2.0478,  0.6097,  ..., -0.8480,  2.9749,  1.6722]],
       grad_fn=<MmBackward0>)


## 3.4 — Autoatenção com pesos treináveis

Agora entram as três matrizes treináveis  `W_query`, `W_key`, `W_value`  que é o mecanismo usado de fato no transformer. Os escores passam a ser divididos por √d_k antes do softmax (daí o nome "scaled"), o que evita que o softmax colapse quando a dimensão de embedding é grande.

In [ ]:
import torch.nn as nn

x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

query_2 = x_2 @ W_query
keys = inputs @ W_key
values = inputs @ W_value

attn_scores_2 = query_2 @ keys.T
d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

In [ ]:
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
print(sa_v1(inputs))


class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        return attn_weights @ values


torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)
print(sa_v2(inputs))

### Aplicando no corpus real

In [ ]:
d_in_real = output_dim
d_out_real = 2

torch.manual_seed(789)
sa_real = SelfAttention_v2(d_in_real, d_out_real)
context_vecs_real = sa_real(input_embeddings)

print(context_vecs_real.shape)
print(context_vecs_real)

## 3.5 — Atenção causal com máscara e dropout

Para gerar texto, o modelo só pode olhar para trás nunca para tokens futuros. A máscara causal zera a atenção para qualquer posição à frente da atual (implementada de duas formas equivalentes: zerar depois do softmax e renormalizar, ou preencher com -inf antes do softmax). O dropout é aplicado por cima, só durante o treino, para reduzir overfitting.

In [ ]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
masked_simple = attn_weights * mask_simple
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print(masked_simple_norm)

In [ ]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=1)
print(attn_weights)

torch.manual_seed(123)
dropout = nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))
print(dropout(attn_weights))

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values


batch = torch.stack((inputs, inputs), dim=0)

torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)

print(context_vecs.shape)
print(context_vecs)

### Aplicando no corpus real

In [ ]:
batch_real = torch.stack((input_embeddings, input_embeddings), dim=0)
print(batch_real.shape)

d_in_real = output_dim
d_out_real = 2

torch.manual_seed(123)
ca_real = CausalAttention(d_in_real, d_out_real, context_length_real, 0.0)
context_vecs_real = ca_real(batch_real)

print(context_vecs_real.shape)
print(context_vecs_real)

## 3.6 — Multi-head attention

Última etapa: rodar várias "cabeças" de atenção causal em paralelo, cada uma aprendendo um aspecto diferente da relação entre os tokens. Primeiro a versão didática (`MultiHeadAttentionWrapper`, que empilha várias `CausalAttention`), depois a versão eficiente (`MultiHeadAttention`, um único conjunto de pesos maior, dividido em cabeças via `.view()`/`.transpose()`).

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


torch.manual_seed(123)
context_length = batch.shape[1]
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print(context_vecs.shape)

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec


torch.manual_seed(123)
batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)

print(context_vecs)
print(context_vecs.shape)

### Aplicando no corpus real

In [ ]:
d_in_real = output_dim
d_out_real = 4

torch.manual_seed(123)
mha_real = MultiHeadAttention(d_in_real, d_out_real, context_length_real, 0.0, num_heads=2)
context_vecs_real = mha_real(batch_real)

print(context_vecs_real)
print(context_vecs_real.shape)